# 06 — Model Evaluation

**AttriSense · Employee Attrition Prediction & Analytics System**

---

## Purpose

Evaluate all four trained classifiers on the held-out test set using a consistent metric suite:

- Accuracy, Precision, Recall, F1, ROC-AUC
- Confusion matrices
- ROC curves
- Feature importance

Select the best model for deployment based on test-set evidence. All evaluation logic lives in `src/attrisense/models/evaluation.py`.

**Prerequisites:** Run `05_Model_Training.ipynb` first (saved `.joblib` pipelines + `training_results.json`).


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from attrisense.config import load_config
from attrisense.models import (
    plot_confusion_matrices,
    plot_feature_importance,
    plot_metrics_comparison,
    plot_roc_curves,
    run_evaluation_pipeline,
)
from attrisense.utils.paths import MODELS_DIR, REPORTS_FIGURES_DIR

config = load_config()
eval_report = run_evaluation_pipeline(config, save_figures=True)

print(f"Test set   : {eval_report.n_test:,} rows ({eval_report.positive_rate*100:.1f}% attrition)")
print(f"Best model : {eval_report.best_model}")
print(f"Results    : {eval_report.results_path}")


## 1. Test-Set Metrics Comparison

Hold-out metrics on the 20% test split (294 rows, ~16% attrition). Models are ranked by ROC-AUC, which was also the cross-validation tuning objective.

**Note:** Accuracy alone is misleading here — the majority class (No attrition) dominates. Precision on the attrition class is low across all models (~0.30–0.43), meaning many flagged employees will not actually leave.


In [ ]:
eval_report.metrics_comparison


In [ ]:
fig = plot_metrics_comparison(eval_report.metrics_comparison)
plt.show()


## 2. Train vs Test Generalization

Tree-based models (Random Forest, XGBoost) achieve near-perfect training scores but degrade substantially on the test set — a sign of overfitting. Logistic Regression generalizes more reliably.


In [ ]:
training = json.loads((MODELS_DIR / "training_results.json").read_text(encoding="utf-8"))

generalization = pd.DataFrame(
    [
        {
            "model": m["name"],
            "train_roc_auc": round(m["train_metrics"]["roc_auc"], 4),
            "test_roc_auc": round(m["test_metrics"]["roc_auc"], 4),
            "auc_gap": round(m["train_metrics"]["roc_auc"] - m["test_metrics"]["roc_auc"], 4),
            "train_f1": round(m["train_metrics"]["f1"], 4),
            "test_f1": round(m["test_metrics"]["f1"], 4),
        }
        for m in training["models"]
    ]
).sort_values("test_roc_auc", ascending=False)

generalization


## 3. Confusion Matrices

Rows = actual, columns = predicted. **Stay** = class 0, **Leave** = class 1.

For attrition screening, recall on the Leave class matters (how many actual leavers are caught). Logistic Regression catches 33 of 47 leavers (70%) but at the cost of 52 false alarms among stayers.


In [ ]:
cm_rows = []
for result in eval_report.results:
    tn, fp, fn, tp = (
        result.confusion_matrix[0, 0],
        result.confusion_matrix[0, 1],
        result.confusion_matrix[1, 0],
        result.confusion_matrix[1, 1],
    )
    cm_rows.append(
        {
            "model": result.model_name,
            "TN (Stay→Stay)": tn,
            "FP (Stay→Leave)": fp,
            "FN (Leave→Stay)": fn,
            "TP (Leave→Leave)": tp,
        }
    )

pd.DataFrame(cm_rows)


In [ ]:
fig = plot_confusion_matrices(eval_report.results)
plt.show()


## 4. ROC Curves

ROC-AUC measures ranking quality — how well the model separates leavers from stayers. None of the models exceed ~0.82 AUC, indicating moderate discriminative ability at best.


In [ ]:
fig = plot_roc_curves(eval_report.results)
plt.show()


## 5. Feature Importance

Top drivers differ by model type: Logistic Regression highlights job role and travel patterns (via one-hot coefficients); tree models emphasize engineered features like `job_stability_index` and `OverTime`.

Importance magnitudes are not directly comparable across model families.


In [ ]:
for result in eval_report.results:
    display(result.feature_importance.head(10))


In [ ]:
for result in eval_report.results:
    fig = plot_feature_importance(result, top_n=15)
    plt.show()


## 6. Final Model Selection

**Selected model: Logistic Regression** (`models/best_model.joblib`)

| Criterion | Logistic Regression | Best Alternative (XGBoost) |
|-----------|--------------------:|---------------------------:|
| Test ROC-AUC | **0.816** | 0.766 |
| Test F1 | **0.500** | 0.338 |
| Test Recall | **0.702** | 0.277 |
| Test Precision | 0.388 | **0.433** |
| Train–Test AUC gap | **0.071** | 0.234 |

**Why Logistic Regression?**
1. **Highest test ROC-AUC (0.816)** — best ranking of attrition risk, aligned with the CV tuning objective.
2. **Best F1 (0.500) and recall (0.702)** — catches ~70% of actual leavers, important for proactive HR intervention.
3. **Most stable generalization** — smallest train–test AUC gap; tree models show severe overfitting (train AUC ≈ 1.0).
4. **Interpretable coefficients** — job role, overtime, income, and travel emerge as clear drivers.

**Caveats (do not overstate performance):**
- ROC-AUC of ~0.82 is **moderate**, not excellent — the model ranks employees better than random but is far from perfect separation.
- Precision is only ~39% — roughly 6 in 10 flagged employees will **not** leave.
- XGBoost has slightly higher accuracy (83%) but misses most leavers (recall 28%) and overfits heavily.
- Results are on a single IBM HR dataset (~1,470 rows); external validation would be needed before production use.

Figures saved to `reports/figures/`. Full results in `models/evaluation_results.json`.


In [ ]:
print(eval_report.selection_rationale)
print(f"\nSaved figures:")
for path in sorted(REPORTS_FIGURES_DIR.glob("*comparison*")) + sorted(REPORTS_FIGURES_DIR.glob("*confusion*")) + sorted(REPORTS_FIGURES_DIR.glob("*roc*")) + sorted(REPORTS_FIGURES_DIR.glob("*feature_importance*")):
    print(f"  {path.name}")
